# Lab 4 — Five features, two dimensions

A two-dimensional bottleneck is asked to carry five sparse features. As the
features become sparser, the model can afford to give each one a direction.
The price is interference: activating one feature also excites its neighbours.

**Milestone:** show that average loss can improve while the one-hot response matrix becomes less diagonal.

In [ ]:
import torch
from uml.superposition import train_model

torch.set_num_threads(4)
results = [train_model(p, steps=1800, seed=1, n=2048) for p in (0.9, 0.3, 0.05)]
for r in results:
    print(f"p={r['p']:.2f}  loss={r['loss']:.4f}  |cos|>0.7: {r['represented']}  cos={r['cosines']}")

p=0.90  loss=0.0674  |cos|>0.7: 2  cos=[-0.706, 0.951, 0.503, -0.227, 0.567]
p=0.30  loss=0.1135  |cos|>0.7: 1  cos=[0.532, 0.178, 0.992, 0.371, 0.659]
p=0.05  loss=0.0054  |cos|>0.7: 3  cos=[0.96, 0.958, -0.939, 0.563, 0.558]

In [ ]:
# The one-hot response matrix: what comes out when exactly one feature is on?
import torch.nn.functional as F
for r in results:
    W, b, D = r["W"], r["b"], r["D"]
    rec = F.relu(D @ W.T + b)
    off = rec - torch.diag(rec.diagonal())
    print(f"p={r['p']:.2f}  diag mean={rec.diagonal().mean():.2f}  off-diagonal mass={off.abs().mean():.2f}")
    print("  ", [[round(float(x), 2) for x in row] for row in rec])

p=0.90  diag mean=0.59  off-diagonal mass=0.39
   [[0.0, 1.22, 0.0, 0.0, 0.63], [0.21, 1.37, 0.0, 0.52, 0.52], [0.56, 0.96, 0.56, 0.6, 0.85], [0.01, 0.55, 0.96, 0.0, 1.17], [0.0, 0.71, 0.22, 0.0, 1.03]]
p=0.30  diag mean=0.41  off-diagonal mass=0.17
   [[0.62, 0.0, 0.0, 0.0, 0.0], [1.05, 0.19, 0.22, 0.0, 0.0], [0.16, 0.49, 0.43, 0.72, 0.0], [0.0, 0.29, 0.28, 0.1, 0.96], [0.0, 0.0, 0.0, 0.0, 0.71]]
p=0.05  diag mean=0.37  off-diagonal mass=0.11
   [[0.71, 0.0, 0.24, 0.0, 0.0], [0.25, 0.69, 0.0, 0.0, 0.0], [0.0, 0.23, 0.0, 0.7, 0.0], [0.0, 0.0, 0.0, 0.25, 0.69], [0.0, 0.0, 0.69, 0.0, 0.22]]

At high sparsity the model represents more of the five features, but each
one-hot input also writes a second feature. The scalar loss is low because
most features are off most of the time; the matrix is where the packing shows.
This is the trade-off the chapter calls superposition: capacity bought with
interference, not a free lunch.

In [ ]:
r = results[-1]
print("feature directions D:")
print(r["D"].round(decimals=2))
print("learned encoder W:")
print(r["W"].round(decimals=2))

feature directions D:
tensor([[ 1.0000,  0.0000],
        [ 0.3100,  0.9500],
        [-0.8100,  0.5900],
        [-0.8100, -0.5900],
        [ 0.3100, -0.9500]])
learned encoder W:
tensor([[ 1.1200,  0.3300],
        [ 0.0300,  1.1700],
        [ 0.6400, -0.9400],
        [-1.0800,  0.3900],
        [-0.7200, -0.9100]])